<a href="https://colab.research.google.com/github/nexageapps/LLM/blob/main/02_Intermediate/L29_Vector_Databases.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L29: Vector Databases - Embeddings and Semantic Search

**Author:** Karthik Arjun  
**LinkedIn:** [Karthik Arjun](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Level:** Intermediate  
**Lesson:** 29 of 30

---

## Learning Objectives

By the end of this lesson, you will:
- Create a vector index with FAISS from document embeddings
- Run approximate nearest-neighbor search for semantic similarity
- Integrate with RAG (L28) for scalable retrieval

---

## Concept: Vector DBs

**Vector databases** store high-dimensional vectors (embeddings) and support fast similarity search (k-NN). Used for RAG, recommendation, deduplication. **FAISS** (Facebook AI) is a library for efficient similarity search; cloud options include Pinecone, Weaviate.

---

In [ ]:
!pip install faiss-cpu sentence-transformers -q
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
import warnings
warnings.filterwarnings('ignore')
print("Libraries loaded.")

## Part 1: Embed and Index with FAISS

---

In [ ]:
encoder = SentenceTransformer("all-MiniLM-L6-v2")
documents = [
    "Paris is the capital of France.",
    "Machine learning enables computers to learn from data.",
    "The Eiffel Tower is in Paris.",
    "Python is popular for data science.",
]

embeddings = encoder.encode(documents)
embeddings = np.array(embeddings).astype("float32")

dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)
print(f"Index built: {index.ntotal} vectors, dim {dim}")

## Part 2: Search

---

In [ ]:
def search(query, index, documents, encoder, top_k=2):
    q_emb = encoder.encode([query])
    q_emb = np.array(q_emb).astype("float32")
    distances, indices = index.search(q_emb, top_k)
    return [documents[i] for i in indices[0]]

results = search("Where is the Eiffel Tower?", index, documents, encoder)
print("Top results:", results)

## Part 3: Approximate Search (IVF)

For large corpora, use an approximate index (e.g. IVF) for faster search.

---

In [ ]:
# For larger datasets: nlist = number of clusters
nlist = 2
quantizer = faiss.IndexFlatL2(dim)
index_ivf = faiss.IndexIVFFlat(quantizer, dim, nlist)
index_ivf.train(embeddings)
index_ivf.add(embeddings)
index_ivf.nprobe = 1
distances, indices = index_ivf.search(encoder.encode(["What is ML?"]).astype("float32"), 2)
print("IVF search:", [documents[i] for i in indices[0]])

## Exercises

1. Index 100+ documents and compare IndexFlatL2 vs IndexIVFFlat search time.
2. Save and load the FAISS index (write_index/read_index) for reuse.
3. Plug this index into the RAG pipeline from L28.

---

## Key Takeaways

1. FAISS provides k-NN search over vectors; IndexFlatL2 is exact, IVF is approximate.
2. Normalize embeddings and use inner product for cosine similarity, or L2 for normalized vectors.
3. Vector DBs scale RAG to large document collections.

---

## Next Lesson

**L30: Project - Document QA System** – End-to-end RAG-based question answering.

---